# Re-order matrix and Pauli decomposition

In [2]:
import numpy as np
import itertools

# Define Pauli matrices
I = np.array([[1, 0], [0, 1]], dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)

paulis = {'I': I, 'X': X, 'Y': Y, 'Z': Z}

def kron_n(mats):
    """Kronecker product of a list of matrices"""
    result = mats[0]
    for m in mats[1:]:
        result = np.kron(result, m)
    return result

def pauli_decomposition(A, N):
    """Decompose a 2^N x 2^N Hermitian matrix into Pauli strings"""
    coeffs = {}
    keys = list(paulis.keys())
    
    for prod in itertools.product(keys, repeat=N):
        label = "".join(prod)
        P = kron_n([paulis[k] for k in prod])
        c = np.trace(P.conj().T @ A) / (2**N)
        if np.abs(c) > 1e-10:  # ignore small numerical noise
            coeffs[label] = c
    
    return coeffs

# Example: decompose a 2-qubit Hermitian matrix
N = 2
A = np.array([[1, 0, 0, 1],
              [0, 0, 1, 0],
              [0, 1, 0, 0],
              [1, 0, 0, -1]], dtype=complex)

decomp = pauli_decomposition(A, N)

print("Pauli decomposition:")
for k, v in decomp.items():
    print(f"{k}: {v}")


Pauli decomposition:
IZ: (0.5+0j)
XX: (1+0j)
ZI: (0.5+0j)


In [5]:
def reorder_columns(A, new_order):
    """
    Reorder the columns of matrix A according to new_order.

    Parameters
    ----------
    A : np.ndarray
        Input matrix (2D).
    new_order : list of int
        New order of column indices. Must be a permutation of [0, 1, ..., n-1].

    Returns
    -------
    np.ndarray
        Matrix with reordered columns.
    """
    return A[:, new_order]


# Example
A = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])

new_order = [2, 0, 1]  # move col 2 first, then col 0, then col 1
B = reorder_columns(A, new_order)

print("Original A:\n", A)
print("Reordered B:\n", B)

Original A:
 [[1 2 3]
 [4 5 6]
 [7 8 9]]
Reordered B:
 [[3 1 2]
 [6 4 5]
 [9 7 8]]


In [33]:
# Create a random 3x3 matrix with values between 0 and 1
N = 2 
random_matrix = np.random.rand(2**N, 2**N)
print("Random matrix:\n", random_matrix)

Random matrix:
 [[3.48987147e-01 7.08859433e-01 7.11480062e-01 3.94721853e-01]
 [9.62474716e-01 2.20597023e-01 4.66029238e-01 5.17328694e-01]
 [2.55509122e-04 3.03453356e-01 7.83974806e-01 9.21472027e-01]
 [5.26707110e-01 3.57834347e-02 3.44685536e-01 6.80712613e-01]]


In [34]:
decomp = pauli_decomposition(random_matrix, N)

print("Pauli decomposition:")
for k, v in decomp.items():
    print(f"{k}: {v}")

Pauli decomposition:
II: (0.5085678972340483+0j)
IX: (0.7343729279727065+0j)
IY: 0.08079280217200321j
IZ: (0.05791307907935189+0j)
XI: (0.3162119249580617+0j)
XX: (0.4227278892358739+0j)
XY: -0.07364028509708645j
XZ: (0.0396558607945286+0j)
YI: 0.2981924530354061j
YX: 0.0076476561423445j
YY: (-0.037986592245476375+0j)
YZ: 0.057419823594797914j
ZI: (-0.2237758121099339+0j)
ZX: (0.10129414672437864+0j)
ZY: -0.20760044347141357j
ZZ: (0.006281982560071786+0j)


In [35]:
new_order = [1,2,3,0]  # move col 2 first, then col 0, then col 1
re_o_random_matrix = reorder_columns(random_matrix, new_order)

print("Original A:\n", random_matrix)
print("Reordered B:\n", re_o_random_matrix)

Original A:
 [[3.48987147e-01 7.08859433e-01 7.11480062e-01 3.94721853e-01]
 [9.62474716e-01 2.20597023e-01 4.66029238e-01 5.17328694e-01]
 [2.55509122e-04 3.03453356e-01 7.83974806e-01 9.21472027e-01]
 [5.26707110e-01 3.57834347e-02 3.44685536e-01 6.80712613e-01]]
Reordered B:
 [[7.08859433e-01 7.11480062e-01 3.94721853e-01 3.48987147e-01]
 [2.20597023e-01 4.66029238e-01 5.17328694e-01 9.62474716e-01]
 [3.03453356e-01 7.83974806e-01 9.21472027e-01 2.55509122e-04]
 [3.57834347e-02 3.44685536e-01 6.80712613e-01 5.26707110e-01]]


In [36]:
decomp = pauli_decomposition(re_o_random_matrix, N)

print("Pauli decomposition:")
for k, v in decomp.items():
    print(f"{k}: {v}")

Pauli decomposition:
II: (0.655766952238835+0j)
IX: (0.40326130195364335+0j)
IY: -0.04739351620105303j
IZ: (0.1593987779058747+0j)
XI: (0.5013338649697454+0j)
XX: (0.42151852023846664+0j)
XY: 0.14496245607493352j
XZ: (-0.15224626083095794+0j)
YI: 0.17726441929180653j
YX: 0.011639399945372997j
YY: (0.22913322949523512+0j)
YZ: -0.13163017090398565j
ZI: (-0.06832261642508342+0j)
ZX: (0.06277724098009912+0j)
ZY: 0.2928350356501048j
ZZ: (-0.03798368032195154+0j)


In [37]:
b = np.random.rand(2**N)
x = np.linalg.solve(random_matrix, b)
re_x = np.linalg.solve(re_o_random_matrix, b)
print("Solution for random_matrix:", x)
print("Solution for re_o_random_matrix:", re_x)

Solution for random_matrix: [ 0.04263211  5.49720286 -6.86261244  4.04577296]
Solution for re_o_random_matrix: [ 5.49720286 -6.86261244  4.04577296  0.04263211]
